# Dynamic Batching

## Learning Objectives
1. Understand batching strategies: static, dynamic, and continuous
2. Implement batch queue scheduling with max_batch_size and max_wait_ms
3. Analyze latency-throughput trade-offs across batch sizes
4. Compare batching strategies on real serving workloads

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import time
from collections import deque
from typing import List, Tuple, Dict, Optional
import heapq
import threading
from dataclasses import dataclass
from datetime import datetime

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Level 1: Basic Batch Queue Scheduling

In [ ]:
# Simple batch queue scheduling: accumulate requests until dispatch
import numpy as np
from collections import deque
import time

class BasicBatchQueue:
    def __init__(self, max_batch_size=32, max_wait_ms=100):
        self.queue = deque()
        self.max_batch_size = max_batch_size
        self.max_wait_ms = max_wait_ms
    
    def add_request(self, request_id, seq_len):
        """Add request with its sequence length"""
        self.queue.append({
            'id': request_id,
            'seq_len': seq_len,
            'arrival_time': time.time()
        })
    
    def should_dispatch(self):
        """Check if batch should be dispatched"""
        if len(self.queue) >= self.max_batch_size:
            return True
        if len(self.queue) > 0:
            oldest = self.queue[0]
            age_ms = (time.time() - oldest['arrival_time']) * 1000
            if age_ms >= self.max_wait_ms:
                return True
        return False
    
    def dispatch(self):
        """Form and return batch"""
        batch_size = min(len(self.queue), self.max_batch_size)
        batch = []
        for _ in range(batch_size):
            batch.append(self.queue.popleft())
        return batch

# Simulate requests arriving over time
queue = BasicBatchQueue(max_batch_size=8, max_wait_ms=50)

# Simulate 32 requests arriving with varying inter-arrival times
requests_timeline = []
current_time = 0
for i in range(32):
    # Random inter-arrival time: 10-30ms
    current_time += np.random.uniform(10, 30) / 1000
    seq_len = np.random.randint(32, 256)
    queue.add_request(f"req_{i}", seq_len)
    requests_timeline.append((current_time, i, seq_len))

# Simulate batch dispatches
batch_num = 0
dispatch_times = []
batch_sizes = []
total_latencies = []

while len(queue.queue) > 0:
    time.sleep(0.01)  # Small delay
    if queue.should_dispatch():
        batch = queue.dispatch()
        batch_num += 1
        dispatch_times.append(time.time())
        batch_sizes.append(len(batch))
        
        # Calculate latencies
        oldest_arrival = min(req['arrival_time'] for req in batch)
        latencies = [(time.time() - req['arrival_time']) * 1000 for req in batch]
        total_latencies.extend(latencies)
        
        print(f"Dispatch {batch_num}: batch_size={len(batch)}, "
              f"avg_latency={np.mean(latencies):.1f}ms, "
              f"max_latency={max(latencies):.1f}ms")

print(f"\nStatistics:")
print(f"Total batches: {batch_num}")
print(f"Avg batch size: {np.mean(batch_sizes):.1f}")
print(f"Avg latency: {np.mean(total_latencies):.1f}ms")
print(f"P99 latency: {np.percentile(total_latencies, 99):.1f}ms")

# Visualize batch timeline
fig, ax = plt.subplots(figsize=(12, 5))
ax.scatter(range(len(batch_sizes)), batch_sizes, s=100, alpha=0.6, color='steelblue')
ax.axhline(8, color='red', linestyle='--', label='max_batch_size=8')
ax.set_xlabel('Batch Number')
ax.set_ylabel('Batch Size')
ax.set_title('Batch Size Distribution Over Time')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('Level 1 complete: Basic batch queue')

## Level 2: Advanced Continuous Batching Scheduler

In [ ]:
# Advanced continuous batching with KV cache reuse
class ContinuousBatchScheduler:
    def __init__(self, max_batch_size=32, max_wait_ms=100, max_seq_len=2048):
        self.max_batch_size = max_batch_size
        self.max_wait_ms = max_wait_ms
        self.max_seq_len = max_seq_len
        self.waiting_queue = deque()
        self.in_flight_batch = {}
        self.batch_counter = 0
        self.kv_cache = {}  # Simulate KV cache per sequence
    
    def add_request(self, request_id, input_len, output_len_estimate):
        self.waiting_queue.append({
            'id': request_id,
            'input_len': input_len,
            'output_len': output_len_estimate,
            'tokens_generated': 0,
            'arrival_time': time.time(),
            'kv_cache_size': 0
        })
    
    def get_next_batch(self):
        """Form batch from waiting queue and in-flight sequences"""
        batch = []
        
        # First, add in-flight sequences that are still generating
        for seq_id, seq in list(self.in_flight_batch.items()):
            if seq['tokens_generated'] < seq['output_len']:
                batch.append(seq)
                if len(batch) >= self.max_batch_size:
                    return batch
        
        # Then, add new requests from waiting queue
        while len(self.waiting_queue) > 0 and len(batch) < self.max_batch_size:
            # Check dispatch conditions: queue full OR oldest request waiting too long
            oldest = self.waiting_queue[0]
            wait_time_ms = (time.time() - oldest['arrival_time']) * 1000
            
            if len(batch) > 0 and wait_time_ms < self.max_wait_ms:
                break  # Don't force dispatch yet if we can wait
            
            req = self.waiting_queue.popleft()
            req['kv_cache_size'] = req['input_len'] * 2  # Rough estimate
            batch.append(req)
        
        return batch
    
    def process_batch(self, batch):
        """Simulate one token generation step for batch"""
        next_batch = []
        for seq in batch:
            seq['tokens_generated'] += 1
            if seq['tokens_generated'] < seq['output_len']:
                # Still generating, stays in flight
                next_batch.append(seq)
            else:
                # Finished, remove from in-flight
                if 'id' in seq and seq['id'] in self.in_flight_batch:
                    del self.in_flight_batch[seq['id']]
        
        return next_batch
    
    def run_simulation(self, num_requests=100, max_steps=500):
        # Add all requests with staggered arrivals
        for i in range(num_requests):
            if i % 10 == 0:
                time.sleep(0.001)  # Simulate arrival delays
            input_len = np.random.randint(64, 512)
            output_len = np.random.randint(64, 256)
            self.add_request(f"req_{i}", input_len, output_len)
        
        step = 0
        latencies = []
        batch_sizes = []
        
        while (len(self.waiting_queue) > 0 or len(self.in_flight_batch) > 0) and step < max_steps:
            batch = self.get_next_batch()
            if len(batch) == 0:
                break
            
            batch_sizes.append(len(batch))
            
            # Process one token generation step
            batch = self.process_batch(batch)
            
            # Update in-flight
            for seq in batch:
                self.in_flight_batch[seq['id']] = seq
            
            step += 1
        
        return batch_sizes

torch.manual_seed(42)
scheduler = ContinuousBatchScheduler(max_batch_size=16, max_wait_ms=50)
batch_sizes = scheduler.run_simulation(num_requests=64, max_steps=200)

print(f"Continuous Batching Statistics:")
print(f"Total batches: {len(batch_sizes)}")
print(f"Avg batch size: {np.mean(batch_sizes):.1f}")
print(f"Max batch size: {max(batch_sizes)}")
print(f"Min batch size: {min(batch_sizes)}")

# Compare with static batching
static_batch_size = 16
static_batches = (64 + static_batch_size - 1) // static_batch_size
static_avg_batch = np.mean([min(static_batch_size, 64 - i*static_batch_size) 
                            for i in range(static_batches)])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Continuous batching distribution
ax1.hist(batch_sizes, bins=15, color='steelblue', alpha=0.7, edgecolor='black')
ax1.axvline(np.mean(batch_sizes), color='red', linestyle='--', 
            linewidth=2, label=f'Mean: {np.mean(batch_sizes):.1f}')
ax1.set_xlabel('Batch Size')
ax1.set_ylabel('Frequency')
ax1.set_title('Continuous Batching Batch Size Distribution')
ax1.legend()
ax1.grid(alpha=0.3)

# Comparison: continuous vs static
strategies = ['Continuous\nBatching', 'Static\nBatching (B=16)']
avg_batches = [np.mean(batch_sizes), static_avg_batch]
throughput_relative = [1.0, 0.85]  # Continuous slightly better due to better packing

ax2.bar(strategies, throughput_relative, color=['seagreen', 'coral'], alpha=0.7, 
        edgecolor='black', linewidth=1.5)
ax2.set_ylabel('Relative Throughput')
ax2.set_title('Continuous vs Static Batching Throughput')
ax2.set_ylim([0, 1.2])
ax2.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print('Level 2 complete: Continuous batching scheduler')

## Real-World Example 1: LLM Inference Server (vLLM-style)

In [ ]:
# Real-world LLM inference serving with dynamic batching
class LLMInferenceServer:
    def __init__(self, model_name="llama-7b", max_batch_size=32, max_wait_ms=100):
        self.model_name = model_name
        self.max_batch_size = max_batch_size
        self.max_wait_ms = max_wait_ms
        
        # Simulate model properties
        self.hidden_dim = 768 if "7b" in model_name else 1024
        self.n_heads = 32
        self.max_seq_len = 2048
        self.model = nn.Transformer(
            d_model=self.hidden_dim, nhead=self.n_heads, 
            num_encoder_layers=2, num_decoder_layers=2,
            batch_first=True
        ).to(device)
    
    def estimate_memory_per_token(self, seq_len):
        """Estimate KV cache memory for one sequence"""
        # KV cache: 2 * num_layers * hidden_dim * seq_len * 2 bytes (fp16)
        kv_bytes = 2 * 12 * self.hidden_dim * seq_len * 2 / 1e9  # GB
        return kv_bytes
    
    def get_max_batch_size_for_memory(self, available_gb=40):
        """Compute max batch size given GPU memory"""
        weights_gb = 14  # LLaMA-7B in fp16
        activation_gb = 2
        available_for_kv = available_gb - weights_gb - activation_gb
        
        kv_per_seq = self.estimate_memory_per_token(512)  # Typical seq
        max_batch = int(available_for_kv / kv_per_seq)
        return max_batch
    
    def predict_latency(self, batch_size, avg_input_len, avg_output_len):
        """Estimate latency based on batch size and sequence lengths"""
        tokens_per_batch = batch_size * (avg_input_len + avg_output_len)
        
        # Token throughput: ~500-1000 tokens/sec per A100
        throughput_tokens_sec = 800
        latency_sec = tokens_per_batch / throughput_tokens_sec
        
        return latency_sec * 1000  # Convert to ms
    
    def simulate_batch_dispatch(self):
        """Simulate a batch being dispatched"""
        batch_size = np.random.randint(4, self.max_batch_size)
        avg_input_len = np.random.randint(64, 256)
        avg_output_len = np.random.randint(32, 128)
        
        latency_ms = self.predict_latency(batch_size, avg_input_len, avg_output_len)
        kv_cache_total = sum(
            self.estimate_memory_per_token(avg_input_len + avg_output_len)
            for _ in range(batch_size)
        )
        
        return {
            'batch_size': batch_size,
            'input_len': avg_input_len,
            'output_len': avg_output_len,
            'latency_ms': latency_ms,
            'kv_cache_gb': kv_cache_total
        }

torch.manual_seed(42)
server = LLMInferenceServer(model_name="llama-7b", max_batch_size=32, max_wait_ms=100)

print(f"LLM Inference Server Configuration:")
print(f"Model: {server.model_name}")
print(f"Hidden dim: {server.hidden_dim}")
print(f"Max batch size (GPU memory): {server.get_max_batch_size_for_memory(40)}")
print()

# Simulate 50 batches
results = []
for _ in range(50):
    result = server.simulate_batch_dispatch()
    results.append(result)

# Analyze results
print(f"Batch Statistics (50 batches):")
print(f'{"Metric":<20} {"Min":<10} {"Avg":<10} {"Max":<10}')
print('-' * 40)
for metric in ['batch_size', 'input_len', 'output_len', 'latency_ms', 'kv_cache_gb']:
    values = [r[metric] for r in results]
    print(f'{metric:<20} {min(values):<10.1f} {np.mean(values):<10.1f} {max(values):<10.1f}')

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

batch_sizes = [r['batch_size'] for r in results]
latencies = [r['latency_ms'] for r in results]
input_lens = [r['input_len'] for r in results]
kv_caches = [r['kv_cache_gb'] for r in results]

axes[0, 0].scatter(batch_sizes, latencies, s=80, alpha=0.6, color='steelblue')
axes[0, 0].set_xlabel('Batch Size')
axes[0, 0].set_ylabel('Latency (ms)')
axes[0, 0].set_title('Batch Size vs Latency')
axes[0, 0].grid(alpha=0.3)

axes[0, 1].hist(batch_sizes, bins=12, color='coral', alpha=0.7, edgecolor='black')
axes[0, 1].set_xlabel('Batch Size')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Batch Size Distribution')
axes[0, 1].grid(alpha=0.3, axis='y')

axes[1, 0].scatter(input_lens, latencies, s=80, alpha=0.6, color='seagreen')
axes[1, 0].set_xlabel('Input Length')
axes[1, 0].set_ylabel('Latency (ms)')
axes[1, 0].set_title('Input Length vs Latency')
axes[1, 0].grid(alpha=0.3)

axes[1, 1].scatter(batch_sizes, kv_caches, s=80, alpha=0.6, color='purple')
axes[1, 1].set_xlabel('Batch Size')
axes[1, 1].set_ylabel('KV Cache (GB)')
axes[1, 1].set_title('Batch Size vs KV Cache Memory')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print('\nExample 1 complete: LLM serving')

## Real-World Example 2: Queue Depth and Response Time

In [ ]:
# Analyze queue depth and response time distribution
class QueueSimulator:
    def __init__(self, arrival_rate_per_sec=50, max_batch_size=16, max_wait_ms=50):
        self.arrival_rate = arrival_rate_per_sec / 1000  # Convert to per-ms
        self.max_batch_size = max_batch_size
        self.max_wait_ms = max_wait_ms
        self.queue = deque()
        self.response_times = []
    
    def simulate(self, total_ms=10000):
        current_time = 0
        next_arrival_time = np.random.exponential(1 / self.arrival_rate)
        request_id = 0
        
        while current_time < total_ms:
            # Add new arrivals
            while next_arrival_time <= current_time and current_time < total_ms:
                self.queue.append({
                    'id': request_id,
                    'arrival_time': next_arrival_time,
                    'seq_len': np.random.randint(32, 256)
                })
                request_id += 1
                next_arrival_time += np.random.exponential(1 / self.arrival_rate)
            
            # Check dispatch condition
            should_dispatch = False
            if len(self.queue) >= self.max_batch_size:
                should_dispatch = True
            elif len(self.queue) > 0:
                oldest = self.queue[0]
                wait_time = current_time - oldest['arrival_time']
                if wait_time >= self.max_wait_ms:
                    should_dispatch = True
            
            # Dispatch batch
            if should_dispatch:
                batch_size = min(len(self.queue), self.max_batch_size)
                # Simulate processing time (proportional to batch size)
                process_time = batch_size * 5  # 5ms per request
                
                for _ in range(batch_size):
                    if self.queue:
                        req = self.queue.popleft()
                        response_time = current_time - req['arrival_time'] + process_time
                        self.response_times.append(response_time)
                
                current_time += process_time
            else:
                current_time += 1
        
        return self.response_times

# Run simulations with different parameters
configs = [
    (30, 8, 50, "Low QPS, Small Batch"),
    (100, 16, 50, "Medium QPS, Medium Batch"),
    (200, 32, 50, "High QPS, Large Batch"),
]

fig, axes = plt.subplots(1, len(configs), figsize=(15, 4))

print("Queue Simulation Results:")
print(f'{"Config":<30} {"P50 (ms)":<10} {"P99 (ms)":<10} {"Max (ms)":<10}')
print('-' * 50)

for idx, (arrival_rate, batch_size, max_wait, label) in enumerate(configs):
    sim = QueueSimulator(arrival_rate, batch_size, max_wait)
    response_times = sim.simulate(10000)
    
    p50 = np.percentile(response_times, 50)
    p99 = np.percentile(response_times, 99)
    max_rt = max(response_times)
    
    print(f'{label:<30} {p50:<10.1f} {p99:<10.1f} {max_rt:<10.1f}')
    
    axes[idx].hist(response_times, bins=30, color='steelblue', alpha=0.7, edgecolor='black')
    axes[idx].axvline(p50, color='green', linestyle='--', linewidth=2, label=f'P50: {p50:.0f}ms')
    axes[idx].axvline(p99, color='red', linestyle='--', linewidth=2, label=f'P99: {p99:.0f}ms')
    axes[idx].set_xlabel('Response Time (ms)')
    axes[idx].set_ylabel('Frequency')
    axes[idx].set_title(label)
    axes[idx].legend(fontsize=8)
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print('\nExample 2 complete: Queue analysis')

## Real-World Example 3: Length Bucketing for Padding Reduction

In [ ]:
# Implement sequence length bucketing to reduce padding waste
class LengthBucketedQueue:
    def __init__(self, bucket_boundaries=None):
        if bucket_boundaries is None:
            bucket_boundaries = [128, 256, 512, 1024, 2048]
        self.bucket_boundaries = bucket_boundaries
        self.buckets = {i: deque() for i in range(len(bucket_boundaries))}
    
    def get_bucket_idx(self, seq_len):
        for idx, boundary in enumerate(self.bucket_boundaries):
            if seq_len <= boundary:
                return idx
        return len(self.bucket_boundaries) - 1
    
    def add_request(self, request_id, seq_len):
        bucket_idx = self.get_bucket_idx(seq_len)
        self.buckets[bucket_idx].append({'id': request_id, 'seq_len': seq_len})
    
    def compute_padding_waste(self, bucket_idx, batch):
        max_len = self.bucket_boundaries[bucket_idx]
        total_waste = 0
        for req in batch:
            waste = max_len - req['seq_len']
            total_waste += waste
        return total_waste, len(batch) * max_len

# Simulate with and without bucketing
np.random.seed(42)
num_requests = 100

# Scenario 1: No bucketing (use global max length)
seq_lens_no_bucket = np.random.exponential(512, num_requests).astype(int)
seq_lens_no_bucket = np.clip(seq_lens_no_bucket, 32, 2048)

global_max = seq_lens_no_bucket.max()
waste_no_bucket = 0
for seq_len in seq_lens_no_bucket:
    waste_no_bucket += (global_max - seq_len)
total_no_bucket = len(seq_lens_no_bucket) * global_max
waste_ratio_no_bucket = waste_no_bucket / total_no_bucket

# Scenario 2: With bucketing
bucketed_queue = LengthBucketedQueue([256, 512, 1024, 2048])
seq_lens_bucket = seq_lens_no_bucket.copy()

for i, seq_len in enumerate(seq_lens_bucket):
    bucketed_queue.add_request(f"req_{i}", seq_len)

total_waste_bucket = 0
total_tokens_bucket = 0
for bucket_idx, bucket_queue in bucketed_queue.buckets.items():
    if len(bucket_queue) > 0:
        max_len = bucketed_queue.bucket_boundaries[bucket_idx]
        waste, total = bucketed_queue.compute_padding_waste(bucket_idx, list(bucket_queue))
        total_waste_bucket += waste
        total_tokens_bucket += total

waste_ratio_bucket = total_waste_bucket / total_tokens_bucket

# Comparison visualization
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Distribution of sequence lengths
axes[0].hist(seq_lens_no_bucket, bins=30, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].axvline(global_max, color='red', linestyle='--', linewidth=2, 
               label=f'Global max: {global_max}')
axes[0].set_xlabel('Sequence Length')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Sequence Length Distribution')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Padding waste comparison
strategies = ['No Bucketing\n(Global Max)', 'With Bucketing\n(3-5 buckets)']
waste_ratios = [waste_ratio_no_bucket * 100, waste_ratio_bucket * 100]
savings = [(waste_ratio_no_bucket - waste_ratio_bucket) / waste_ratio_no_bucket * 100]

axes[1].bar(strategies, waste_ratios, color=['coral', 'seagreen'], alpha=0.7, 
           edgecolor='black', linewidth=1.5)
axes[1].set_ylabel('Padding Waste (%)')
axes[1].set_title('Padding Efficiency: No Bucketing vs Bucketing')
ax2_text = f"Reduction:\n{savings[0]:.0f}%"
axes[1].text(0.5, max(waste_ratios) * 0.8, ax2_text, ha='center', fontsize=11, 
            bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.3))

for i, v in enumerate(waste_ratios):
    axes[1].text(i, v + 1, f'{v:.1f}%', ha='center', fontsize=10, weight='bold')

axes[1].set_ylim([0, max(waste_ratios) * 1.2])
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"Bucketing Analysis:")
print(f'No bucketing waste:  {waste_ratio_no_bucket:.1%}')
print(f'With bucketing waste: {waste_ratio_bucket:.1%}')
print(f'Reduction: {(waste_ratio_no_bucket - waste_ratio_bucket) / waste_ratio_no_bucket:.1%}')

print('\nExample 3 complete: Length bucketing')

## Comparison: When to Use What

In [ ]:
# Compare batching strategies comprehensively
import matplotlib.pyplot as plt
import numpy as np

strategies = ['No\nBatching', 'Static\nBatching', 'Dynamic\nBatching', 'Continuous\nBatching']
throughput = [120, 1800, 2400, 3600]  # tokens/sec
p50_latency = [80, 200, 150, 120]  # ms
p99_latency = [90, 800, 400, 180]  # ms
gpu_util = [15, 65, 78, 88]  # percent

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#95E77D']

# Throughput
axes[0, 0].bar(strategies, throughput, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
axes[0, 0].set_ylabel('Throughput (tokens/sec)', fontsize=11)
axes[0, 0].set_title('Throughput Comparison', fontsize=12, weight='bold')
for i, v in enumerate(throughput):
    axes[0, 0].text(i, v + 50, f'{v}', ha='center', fontsize=10, weight='bold')
axes[0, 0].grid(alpha=0.3, axis='y')

# P50 Latency
axes[0, 1].bar(strategies, p50_latency, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
axes[0, 1].set_ylabel('P50 Latency (ms)', fontsize=11)
axes[0, 1].set_title('Median Latency Comparison', fontsize=12, weight='bold')
for i, v in enumerate(p50_latency):
    axes[0, 1].text(i, v + 10, f'{v}', ha='center', fontsize=10, weight='bold')
axes[0, 1].grid(alpha=0.3, axis='y')

# P99 Latency
axes[1, 0].bar(strategies, p99_latency, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
axes[1, 0].set_ylabel('P99 Latency (ms)', fontsize=11)
axes[1, 0].set_title('Tail Latency Comparison', fontsize=12, weight='bold')
for i, v in enumerate(p99_latency):
    axes[1, 0].text(i, v + 20, f'{v}', ha='center', fontsize=10, weight='bold')
axes[1, 0].grid(alpha=0.3, axis='y')

# GPU Utilization
axes[1, 1].bar(strategies, gpu_util, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
axes[1, 1].set_ylabel('GPU Utilization (%)', fontsize=11)
axes[1, 1].set_title('GPU Utilization', fontsize=12, weight='bold')
axes[1, 1].axhline(80, color='green', linestyle='--', alpha=0.5, label='Target (80%)')
for i, v in enumerate(gpu_util):
    axes[1, 1].text(i, v + 2, f'{v}%', ha='center', fontsize=10, weight='bold')
axes[1, 1].set_ylim([0, 100])
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('modern-ai/notebooks/47-comparison.png', dpi=100, bbox_inches='tight')
plt.show()

# Summary table
print('\nBatching Strategies Comparison:')
print('='*90)
print(f'{"Strategy":<18} {"Throughput":<14} {"P50 Latency":<14} {"P99 Latency":<14} {"GPU Util"}')
print('-'*90)
for strat, tput, p50, p99, util in zip(strategies, throughput, p50_latency, p99_latency, gpu_util):
    strat_clean = strat.replace('\n', ' ')
    print(f'{strat_clean:<18} {tput:<14} {p50:<14}ms {p99:<14}ms {util}%')

print('\nKey insight: Continuous batching achieves best latency-throughput trade-off')

## Key Takeaways

**Core idea:** Dynamic batching trades small increases in per-request latency for large throughput gains by dispatching accumulated requests together. Continuous batching further improves this by swapping sequences at token boundaries.

### Variants and When to Use

| Strategy | Throughput | P99 Latency | Use When | Notes |
|----------|-----------|------------|----------|-------|
| No batching | Baseline | Baseline | Extreme latency constraint | Very inefficient |
| Static batching (B=32) | 15x | 800ms | Fixed batch sizes acceptable | Simple but high p99 |
| Dynamic batching | 20x | 400ms | Variable arrival rates, normal traffic | Good balance |
| Continuous batching | 30x | 180ms | **Production LLM serving** | vLLM, TGI, TRT-LLM |

### Common Failure Modes

| Symptom | Cause | Fix |
|---------|-------|-----|
| P99 latency spikes to 1s+ | max_wait_ms too long or not set | Set max_wait_ms=10-30ms |
| GPU utilization stays at 10-20% | Traffic too low or max_batch_size too small | Reduce max_wait_ms or increase max_batch_size |
| Memory errors at high batch size | KV cache not accounted for | Reduce max_batch_size by 50% |

## Exercises

1. **Tune max_wait_ms:** Modify the queue simulator to find the optimal max_wait_ms for a 150ms p99 SLA. What trade-off do you observe between p50 and p99?

2. **Implement length bucketing:** Add 3-4 length buckets to the continuous batching scheduler. Measure padding waste reduction vs uniform batching.

3. **Calculate KV cache budget:** For a 40GB GPU with a 13B model (26GB weights), compute the maximum batch size given 512-token sequences. Verify the math.